# Rewrite fcst_joined_timeseries with smaller row groups

One-off `rewrite-all` of `fcst_joined_timeseries` so its existing files pick up
`write.parquet.row-group-size-bytes = 16 MiB`.

**Why.** Parquet row groups are the smallest unit a reader can split a file on. The weekly
`routine_table_maintenance` compaction writes 512 MB files using the table's write properties, and
until now this table had none, so it got Iceberg's 128 MB default. On 2026-10-04, the first full
NWMD metrics scan after a compaction (3 years of `nwm30_medium_range`, ~11 GB) got **88 tasks**
instead of the ~600 it had before, so about two thirds of a 64-executor cluster sat idle through
the scan stage. Iceberg's adaptive split sizing wanted 16 MB splits but cannot cut below a row
group.

**Where the property comes from.** `fcst_joined_timeseries` is not managed by
`warehouse_migrations/`. It is created and updated by the joined-forecast Prefect flows, and
`ensure_joined_forecast_table_properties()` in
`prefect-workflows/workflows/metrics/utils/joined_forecast_utils.py` sets the property after every
write. This notebook does **not** set it. It refuses to run until that change is deployed and a
flow run has applied it, so there is one source of truth.

**Why a manual rewrite.** The property only affects files written after it is set. The weekly
compaction only rewrites files outside its size band (about 75-180% of 512 MB), so the files it has
already compacted would keep 128 MB row groups indefinitely.

Chunked by `secondary_configuration_name`, largest first, with partial progress, following
`08_rewrite_uncompacted_timeseries.ipynb`. Reads are unaffected while this runs (snapshot isolation).
Avoid overlapping with the joined-forecast flows: a rewrite group whose files a concurrent upsert
touches fails to commit and has to be redone.


In [1]:
import time

from teehr.evaluation.spark_session_utils import create_spark_session

TABLE = "fcst_joined_timeseries"
FQ = f"iceberg.teehr.{TABLE}"
ROW_GROUP_BYTES = str(16 * 1024 * 1024)  # must match JOINED_FORECAST_TABLE_PROPERTIES

In [2]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=16,
    executor_memory="24g",
    executor_cores=4
)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:📦 Configuring Spark cluster with container image: None
INFO:teehr.evaluation.spark_session_utils:🔍 Initial spark namespace from ENV: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔍 Connecting to Kubernetes API: https://172.20.0.1:443
INFO:teehr.evaluation.spark_session_utils:🎯 Executor namespace: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔐 Executor service account: spark (in teehr-hub)
INFO:teehr.evaluation.spark_session_utils:🔐 Using in-cluster authentication
INFO:teehr.evaluation.spark_session_utils:🔗 Setting driver host to pod IP: 10.0.0.19
INFO:teehr.evaluation.spark_session_utils:✅ Spark cluster configuration successful!
INFO:teehr.evaluation.spark_session_utils:   - Executor instances: 16
INFO:teehr.evaluation.spark_session_utils:   - Executor memory: 24g
INFO:teehr.evaluation.spark_session_utils:   - Executor cores: 4
INFO:teehr.evaluatio

In [3]:
props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {FQ}").collect()}
if props.get("write.parquet.row-group-size-bytes") != ROW_GROUP_BYTES:
    raise RuntimeError(
        f"{FQ} has write.parquet.row-group-size-bytes="
        f"{props.get('write.parquet.row-group-size-bytes')!r}, expected {ROW_GROUP_BYTES}. "
        "Deploy the ensure_joined_forecast_table_properties change and let a "
        "joined-forecast flow run apply it. Rewriting now would just rewrite the "
        "same 128 MB row groups."
    )
if "sort-order" not in props:
    raise RuntimeError(f"{FQ} has no declared sort order; strategy => 'sort' needs one.")
print(f"row group size: {props['write.parquet.row-group-size-bytes']}")
print(f"sort order: {props['sort-order']}")

row group size: 16777216
sort order: location_id ASC NULLS LAST, secondary_location_id ASC NULLS LAST, reference_time ASC NULLS LAST, value_time ASC NULLS LAST


In [4]:
MAX_CONCURRENT_GROUPS = 100
MAX_COMMITS = 50


def rewrite(where=None):
    """Sort-rewrite the table, optionally scoped to a subset of partitions.

    rewrite-all because the files are already at target size: without it the
    compaction leaves them alone and they keep their 128 MB row groups. No
    sort_order argument, so the table's declared order is used.
    """
    scope = f", where => \"{where}\"" if where else ""
    return spark.sql(f"""
        CALL iceberg.system.rewrite_data_files(
            table => 'teehr.{TABLE}',
            strategy => 'sort'{scope},
            options => map(
                'rewrite-all', 'true',
                'partial-progress.enabled', 'true',
                'partial-progress.max-commits', '{MAX_COMMITS}',
                'max-concurrent-file-group-rewrites', '{MAX_CONCURRENT_GROUPS}'
            )
        )
    """).collect()[0]


def report():
    f = spark.sql(f"""
        SELECT count(*) AS files,
               round(sum(file_size_in_bytes) / 1e9, 2) AS gb,
               round(avg(file_size_in_bytes) / 1e6, 1) AS avg_mb,
               round(avg(cardinality(split_offsets)), 2) AS row_groups_per_file
        FROM {FQ}.files
    """).collect()[0]
    print(f"{TABLE}: {f['files']} files, {f['gb']} GB, {f['avg_mb']} MB avg, "
          f"{f['row_groups_per_file']} row groups/file")

## Before

At 512 MB files with 128 MB row groups, expect about 4 row groups per file. After the rewrite,
expect about 30.

In [5]:
report()

fcst_joined_timeseries: 452 files, 41.15 GB, 91.0 MB avg, 2.11 row groups/file


## Rewrite, one configuration at a time

`rewrite_data_files` has no resume memory, and re-running with `rewrite-all` redoes groups that
already finished. Stop between chunks freely, and add finished names to `ALREADY_DONE` after a
kernel restart.

In [6]:
rows = spark.sql(f"""
    SELECT partition.secondary_configuration_name AS cfg,
           round(sum(total_data_file_size_in_bytes) / 1e9, 2) AS gb
    FROM {FQ}.partitions
    GROUP BY 1 ORDER BY 2 DESC
""").collect()
# A NULL here would be data that chunking silently skips.
if any(r["cfg"] is None for r in rows):
    raise RuntimeError(
        "Some partitions have no secondary_configuration_name. Rewrite the table "
        "in one unscoped call instead: rewrite()."
    )
CONFIGS = [(r["cfg"], r["gb"]) for r in rows]
print(f"{len(CONFIGS)} configurations, {sum(g for _, g in CONFIGS):.1f} GB total")
for cfg, gb in CONFIGS:
    print(f"    {cfg}: {gb} GB")

21 configurations, 41.1 GB total
    nwm30_medium_range_blend: 9.55 GB
    nwm30_medium_range: 9.45 GB
    nwm30_short_range: 6.68 GB
    nrds_v22_cfenom_medium_range: 5.39 GB
    nrds_v22_lstm0_medium_range: 4.19 GB
    nrds_v22_cfenom_short_range: 2.43 GB
    nrds_v22_lstm0_short_range: 2.15 GB
    nwm31_medium_range: 0.44 GB
    nwm31_medium_range_blend: 0.43 GB
    nwm31_short_range: 0.3 GB
    nwm30_medium_range_alaska: 0.03 GB
    nwpsrfc_streamflow_forecast: 0.03 GB
    nwm30_medium_range_blend_alaska: 0.03 GB
    nwm30_short_range_alaska: 0.01 GB
    nwm30_short_range_puertorico: 0.01 GB
    nwm30_short_range_hawaii: 0.01 GB
    nwm31_medium_range_alaska: 0.0 GB
    nwm31_short_range_alaska: 0.0 GB
    nwm31_short_range_hawaii: 0.0 GB
    nwm31_medium_range_blend_alaska: 0.0 GB
    nwm31_short_range_puertorico: 0.0 GB


In [7]:
# Seed this by hand to resume after a kernel restart. Within a session the
# loop also remembers what it finished.
ALREADY_DONE = set()
DONE = globals().get("DONE", set()) | ALREADY_DONE

for cfg, gb in CONFIGS:
    if cfg in DONE:
        print(f"skip {cfg} ({gb} GB) - already done")
        continue
    print(f"--- {cfg} ({gb} GB)", flush=True)
    started = time.time()
    r = rewrite(where=f"secondary_configuration_name = '{cfg}'")
    print(f"    {r['rewritten_data_files_count']} -> {r['added_data_files_count']} files, "
          f"failed={r['failed_data_files_count']}, {time.time() - started:.0f}s", flush=True)
    DONE.add(cfg)

--- nwm30_medium_range_blend (9.55 GB)
    36 -> 36 files, failed=0, 1642s
--- nwm30_medium_range (9.45 GB)
    36 -> 36 files, failed=0, 1418s
--- nwm30_short_range (6.68 GB)
    36 -> 36 files, failed=0, 631s
--- nrds_v22_cfenom_medium_range (5.39 GB)
    20 -> 17 files, failed=0, 756s
--- nrds_v22_lstm0_medium_range (4.19 GB)
    15 -> 13 files, failed=0, 751s
--- nrds_v22_cfenom_short_range (2.43 GB)
    12 -> 12 files, failed=0, 394s
--- nrds_v22_lstm0_short_range (2.15 GB)
    9 -> 9 files, failed=0, 379s
--- nwm31_medium_range (0.44 GB)
    7 -> 3 files, failed=0, 1050s
--- nwm31_medium_range_blend (0.43 GB)
    7 -> 3 files, failed=0, 1031s
--- nwm31_short_range (0.3 GB)
    4 -> 3 files, failed=0, 465s
--- nwm30_medium_range_alaska (0.03 GB)
    19 -> 19 files, failed=0, 20s
--- nwpsrfc_streamflow_forecast (0.03 GB)
    9 -> 9 files, failed=0, 17s
--- nwm30_medium_range_blend_alaska (0.03 GB)
    22 -> 19 files, failed=0, 14s
--- nwm30_short_range_alaska (0.01 GB)
    41 -> 18

## After

Snapshot and orphan-file cleanup is left to `routine_table_maintenance`. The pre-rewrite files stay
on S3 until their snapshots expire.

In [8]:
report()

fcst_joined_timeseries: 400 files, 38.88 GB, 97.2 MB avg, 7.12 row groups/file


In [9]:
spark.stop()